# 04 — Hybrid Model Development & Ensemble Evaluation

**Phase 4: Hybrid ML + DL**  
Combines Phase 1 Random Forest & XGBoost (base tree models) with Phase 2 MLP (deep learning) in two ensemble strategies:

1. **Weighted Ensemble**: Linear weighted average of RF, XGB, and base MLP predictions.
2. **Hybrid (RF+XGB+ResidualMLP)**: Tree ensemble base + a lightweight MLP trained to correct systematic residual errors.

**Pipeline:**
1. Load Phase 1 tree models + Phase 2 base MLP
2. Generate OOF predictions (5-fold CV) with no data leakage
3. Train ResidualMLPNet on OOF residuals
4. Compare all 5 models on val + test sets
5. Export artifacts

## Section 1 — Setup & Imports

In [ ]:
import sys
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

warnings.filterwarnings('ignore')

# ── Resolve project root ───────────────────────────────────────────────────
NOTEBOOK_DIR = Path().resolve()
REPO_ROOT    = NOTEBOOK_DIR.parent  # Phase_2_Deep_Learning → repo root
if not (REPO_ROOT / 'Phase_1_Machine_Learning').exists():
    REPO_ROOT = NOTEBOOK_DIR.parents[1]

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f'REPO_ROOT: {REPO_ROOT}')
print(f'PyTorch:   {torch.__version__}')

In [ ]:
# ── Phase 2 src imports ────────────────────────────────────────────────────
PHASE2_SRC = str(REPO_ROOT / 'Phase_2_Deep_Learning')
if PHASE2_SRC not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'Phase_2_Deep_Learning'))

from Phase_2_Deep_Learning.src.dataset import (
    load_feature_config, load_transformers, prepare_tensors, load_raw_splits
)
from Phase_2_Deep_Learning.src.dl_models import load_model_checkpoint
from Phase_2_Deep_Learning.src.training import get_device, compute_regression_metrics
from Phase_2_Deep_Learning.src.hybrid_models import (
    HybridConfig, ResidualMLPNet, HybridEnsemble, WeightedEnsemble,
    save_hybrid_checkpoint, load_hybrid_checkpoint
)
from Phase_2_Deep_Learning.src.oof_generator import (
    generate_ensemble_oof, compute_residuals, save_oof_artifacts
)
from Phase_2_Deep_Learning.src.residual_trainer import (
    ResidualDataset, build_residual_dataloaders, train_residual_mlp, evaluate_residual_model
)
from Phase_1_Machine_Learning.src.models import load_tree_model

print('✓ All imports successful')

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────
PHASE2_DIR   = REPO_ROOT / 'Phase_2_Deep_Learning'
PHASE1_DIR   = REPO_ROOT / 'Phase_1_Machine_Learning'
DATA_DIR     = REPO_ROOT / 'data' / 'GeneratedOutput' / '02.Preprocessing&FeatureSelection'
OUTPUT_DIR   = REPO_ROOT / 'data' / 'GeneratedOutput' / '04.HybridModel'
MODELS_DIR   = PHASE2_DIR / 'models'
ARTIFACTS_DIR= PHASE2_DIR / 'artifacts'
CONFIGS_DIR  = PHASE2_DIR / 'configs'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

RF_MODEL_PATH  = REPO_ROOT / 'data' / 'GeneratedOutput' / '03.ModelTraining' / 'random_forest_model.joblib'
XGB_MODEL_PATH = REPO_ROOT / 'data' / 'GeneratedOutput' / '03.ModelTraining' / 'xgboost_model.json'
MLP_CKPT_PATH  = MODELS_DIR / 'mlp_best_checkpoint.pt'

print(f'Output dir  : {OUTPUT_DIR}')
print(f'RF model    : {RF_MODEL_PATH.exists()}')
print(f'XGB model   : {XGB_MODEL_PATH.exists()}')
print(f'MLP ckpt    : {MLP_CKPT_PATH.exists()}')

In [ ]:
# ── Feature config & transformers ─────────────────────────────────────────
feature_cfg = load_feature_config(CONFIGS_DIR / 'feature_config.json')
scaler, encoder = load_transformers(ARTIFACTS_DIR)

NUM_FEATURES = feature_cfg['numerical_features']
CAT_FEATURES = feature_cfg['categorical_features']
TARGET_COL   = feature_cfg['target_variable']

print(f'Numerical features : {len(NUM_FEATURES)}')
print(f'Categorical features: {len(CAT_FEATURES)}')
print(f'Target             : {TARGET_COL}')

In [ ]:
# ── Load data splits ───────────────────────────────────────────────────────
train_df, val_df, test_df = load_raw_splits(DATA_DIR)
print(f'Train: {len(train_df):,} rows | Val: {len(val_df):,} rows | Test: {len(test_df):,} rows')

In [ ]:
# ── Load Phase 1 tree models ───────────────────────────────────────────────
rf_model  = load_tree_model(str(RF_MODEL_PATH),  model_type='rf')
xgb_model = load_tree_model(str(XGB_MODEL_PATH), model_type='xgb')
print(f'✓ RF model loaded  : {type(rf_model).__name__}')
print(f'✓ XGB model loaded : {type(xgb_model).__name__}')

In [ ]:
# ── Prepare tensors for all splits ────────────────────────────────────────
DEVICE = get_device()
print(f'Device: {DEVICE}')

x_train_num, x_train_cat, x_train_all, y_train_t = prepare_tensors(
    train_df, scaler, encoder, NUM_FEATURES, CAT_FEATURES, TARGET_COL
)
x_val_num, x_val_cat, x_val_all, y_val_t = prepare_tensors(
    val_df, scaler, encoder, NUM_FEATURES, CAT_FEATURES, TARGET_COL
)
x_test_num, x_test_cat, x_test_all, y_test_t = prepare_tensors(
    test_df, scaler, encoder, NUM_FEATURES, CAT_FEATURES, TARGET_COL
)

X_train_np = x_train_all.numpy()
y_train_np = y_train_t.numpy().flatten()
X_val_np   = x_val_all.numpy()
y_val_np   = y_val_t.numpy().flatten()
X_test_np  = x_test_all.numpy()
y_test_np  = y_test_t.numpy().flatten()

print(f'Train features: {X_train_np.shape} | Targets: {y_train_np.shape}')
print(f'Val   features: {X_val_np.shape}   | Targets: {y_val_np.shape}')
print(f'Test  features: {X_test_np.shape}  | Targets: {y_test_np.shape}')

## Section 2 — Out-of-Fold Predictions

In [ ]:
# ── Generate OOF predictions (5-fold KFold, no data leakage) ─────────────
oof_rf, oof_xgb, fold_metrics_dict = generate_ensemble_oof(
    X_train_np, y_train_np, n_splits=5, random_state=42
)
print(f'\nOOF RF  shape : {oof_rf.shape}')
print(f'OOF XGB shape : {oof_xgb.shape}')

In [ ]:
# ── Print OOF metrics table ────────────────────────────────────────────────
def print_oof_table(fold_metrics_dict):
    header = f"{'Model':<6} {'Fold':<6} {'MAE':>8} {'RMSE':>8} {'R²':>8}"
    print('=' * 45)
    print(header)
    print('-' * 45)
    for model_name, folds in fold_metrics_dict.items():
        maes, rmses, r2s = [], [], []
        for m in folds:
            print(f"{model_name.upper():<6} {m['fold']:<6} {m['mae']:>8.4f} {m['rmse']:>8.4f} {m['r2']:>8.4f}")
            maes.append(m['mae']); rmses.append(m['rmse']); r2s.append(m['r2'])
        print('-' * 45)
        print(f"{'':6} {'mean':<6} {np.mean(maes):>8.4f} {np.mean(rmses):>8.4f} {np.mean(r2s):>8.4f}")
        print(f"{'':6} {'std':<6} {np.std(maes):>8.4f} {np.std(rmses):>8.4f} {np.std(r2s):>8.4f}")
        print('=' * 45)

print_oof_table(fold_metrics_dict)

In [ ]:
# ── Compute residuals ──────────────────────────────────────────────────────
residuals_rf,  stats_rf  = compute_residuals(y_train_np, oof_rf)
residuals_xgb, stats_xgb = compute_residuals(y_train_np, oof_xgb)

print('RF  residual stats :', stats_rf)
print('XGB residual stats :', stats_xgb)

In [ ]:
# ── Save OOF artifacts ─────────────────────────────────────────────────────
save_oof_artifacts(oof_rf, oof_xgb, residuals_rf, residuals_xgb, y_train_np, OUTPUT_DIR)

In [ ]:
# ── Residual distribution plot ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, residuals, name, color in zip(
    axes, [residuals_rf, residuals_xgb], ['RF', 'XGB'], ['#2196F3', '#FF9800']
):
    ax.hist(residuals, bins=60, color=color, alpha=0.7, edgecolor='white')
    ax.axvline(0, color='black', lw=1.5, ls='--', label='Zero')
    ax.axvline(np.mean(residuals), color='red', lw=1.5, ls='-', label=f'Mean={np.mean(residuals):.2f}')
    ax.set_title(f'{name} OOF Residual Distribution', fontsize=13, fontweight='bold')
    ax.set_xlabel('Residual (y_true − ŷ_base)')
    ax.set_ylabel('Count')
    ax.legend(fontsize=10)
    stats = stats_rf if name == 'RF' else stats_xgb
    ax.text(0.98, 0.95,
            f"std={stats['std']:.2f}\nmax_abs={stats['max_abs']:.2f}\n%within5={stats['pct_within_5']:.1f}%",
            transform=ax.transAxes, ha='right', va='top', fontsize=9,
            bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.8))

plt.suptitle('OOF Residual Distributions — RF vs XGBoost', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'oof_residual_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Residual distribution plot saved')

## Section 3 — Residual MLP Architecture

In [ ]:
# ── Instantiate HybridConfig & ResidualMLPNet ──────────────────────────────
hybrid_cfg = HybridConfig(
    input_dim   = 53,
    hidden_dims = [256, 128, 64],
    dropout_rate= 0.2,
    use_batch_norm=True,
    rf_weight   = 0.4,
    xgb_weight  = 0.4,
    mlp_weight  = 0.2,
)

residual_net = ResidualMLPNet(hybrid_cfg).to(DEVICE)
print(residual_net.architecture_summary())

In [ ]:
# ── Verify forward pass with dummy input ──────────────────────────────────
dummy_in  = torch.zeros(4, 54).to(DEVICE)   # 53 features + 1 base pred
dummy_out = residual_net(dummy_in)
print(f'Input shape  : {dummy_in.shape}')
print(f'Output shape : {dummy_out.shape}')
assert dummy_out.shape == (4, 1), f'Expected (4,1), got {dummy_out.shape}'
print('✓ Forward pass verified')

## Section 4 — Residual MLP Training

In [ ]:
# ── Prepare base predictions for training (average of OOF RF + XGB) ───────
base_train_pred = (oof_rf + oof_xgb) / 2.0          # (N_train,)
residuals_avg   = y_train_np - base_train_pred        # signed residuals

# Validation base predictions (use full-data trained models)
rf_val_pred   = rf_model.predict(X_val_np)
xgb_val_pred  = xgb_model.predict(X_val_np)
base_val_pred = (rf_val_pred + xgb_val_pred) / 2.0
residuals_val = y_val_np - base_val_pred

print(f'Train base preds shape : {base_train_pred.shape}')
print(f'Train residuals  — mean: {residuals_avg.mean():.4f}, std: {residuals_avg.std():.4f}')
print(f'Val   residuals  — mean: {residuals_val.mean():.4f}, std: {residuals_val.std():.4f}')

In [ ]:
# ── Build ResidualDataset and DataLoaders ──────────────────────────────────
base_train_t  = torch.from_numpy(base_train_pred.astype(np.float32))
res_train_t   = torch.from_numpy(residuals_avg.astype(np.float32))
base_val_t    = torch.from_numpy(base_val_pred.astype(np.float32))
res_val_t     = torch.from_numpy(residuals_val.astype(np.float32))

train_res_loader, val_res_loader = build_residual_dataloaders(
    x_train_all, base_train_t, res_train_t,
    x_val_all,   base_val_t,   res_val_t,
    batch_size=1024,
)
print(f'Train loader: {len(train_res_loader)} batches')
print(f'Val   loader: {len(val_res_loader)} batches')

In [ ]:
# ── Train ResidualMLPNet ───────────────────────────────────────────────────
RESIDUAL_CKPT_PATH = MODELS_DIR / 'residual_mlp_checkpoint.pt'

res_history = train_residual_mlp(
    model        = residual_net,
    train_loader = train_res_loader,
    val_loader   = val_res_loader,
    n_epochs     = 50,
    lr           = 1e-3,
    patience     = 10,
    device       = DEVICE,
    save_path    = RESIDUAL_CKPT_PATH,
)
print(f"\nBest Val MAE: {res_history['best_val_mae']:.4f} @ epoch {res_history['best_epoch']}")

In [ ]:
# ── Training curves ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs = range(1, len(res_history['train_loss']) + 1)

axes[0].plot(epochs, res_history['train_loss'], label='Train Huber Loss', color='#2196F3')
axes[0].plot(epochs, res_history['val_loss'],   label='Val Huber Loss',   color='#F44336')
axes[0].axvline(res_history['best_epoch'], color='green', ls='--', lw=1.5, label=f"Best ep={res_history['best_epoch']}")
axes[0].set_title('ResidualMLP — Training vs Validation Huber Loss', fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss'); axes[0].legend()

axes[1].plot(epochs, res_history['val_mae'], label='Val MAE', color='#9C27B0')
axes[1].axhline(res_history['best_val_mae'], color='green', ls='--', lw=1.5, label=f"Best MAE={res_history['best_val_mae']:.4f}")
axes[1].set_title('ResidualMLP — Validation MAE', fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('MAE'); axes[1].legend()

plt.suptitle('Phase 4 — Residual MLP Training Convergence', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'residual_mlp_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Training curves saved')

## Section 5 — Hybrid Ensemble Evaluation

In [ ]:
# ── Load base MLP ──────────────────────────────────────────────────────────
base_mlp, mlp_ckpt = load_model_checkpoint(MLP_CKPT_PATH, device=DEVICE)
base_mlp.eval()
print(f'✓ Base MLP loaded (config: {mlp_ckpt.get("architecture_config", {}).get("name", "MLP")})')

In [ ]:
# ── Helper: get MLP predictions ────────────────────────────────────────────
@torch.no_grad()
def get_mlp_preds(model, X_tensor, device):
    model.eval()
    X_tensor = X_tensor.to(device)
    preds = model(X_tensor).cpu().numpy().flatten()
    return preds

# ── Compute all base predictions ───────────────────────────────────────────
# Validation set
rf_val   = rf_model.predict(X_val_np)
xgb_val  = xgb_model.predict(X_val_np)
mlp_val  = get_mlp_preds(base_mlp, x_val_all, DEVICE)

# Test set
rf_test  = rf_model.predict(X_test_np)
xgb_test = xgb_model.predict(X_test_np)
mlp_test = get_mlp_preds(base_mlp, x_test_all, DEVICE)

print(f'Val  — RF: {rf_val.shape}, XGB: {xgb_val.shape}, MLP: {mlp_val.shape}')
print(f'Test — RF: {rf_test.shape}, XGB: {xgb_test.shape}, MLP: {mlp_test.shape}')

In [ ]:
# ── Weighted Ensemble predictions ──────────────────────────────────────────
weighted_ens = WeightedEnsemble(rf_weight=0.4, xgb_weight=0.4, mlp_weight=0.2)

we_val  = weighted_ens.predict(rf_val,  xgb_val,  mlp_val)
we_test = weighted_ens.predict(rf_test, xgb_test, mlp_test)

print(f'Weighted Ensemble — Val: {we_val.shape}, Test: {we_test.shape}')

In [ ]:
# ── Hybrid corrected predictions (base_pred + residual correction) ─────────
hybrid_model = HybridEnsemble(hybrid_cfg).to(DEVICE)

# Load residual net weights into hybrid model
hybrid_model.residual_net.load_state_dict(residual_net.state_dict())
hybrid_model.eval()

base_val_pred_avg  = (rf_val  + xgb_val)  / 2.0
base_test_pred_avg = (rf_test + xgb_test) / 2.0

hybrid_val  = hybrid_model.predict_numpy(X_val_np,  base_val_pred_avg,  device=DEVICE)
hybrid_test = hybrid_model.predict_numpy(X_test_np, base_test_pred_avg, device=DEVICE)

print(f'Hybrid Ensemble — Val: {hybrid_val.shape}, Test: {hybrid_test.shape}')

In [ ]:
# ── Comprehensive metrics table ────────────────────────────────────────────
def eval_metrics(y_true, y_pred, label, split):
    m = compute_regression_metrics(y_true, y_pred)
    return {'Model': label, 'Split': split, 'MAE': m['mae'], 'RMSE': m['rmse'], 'R²': m['r2']}

rows = []
for split, y_true, preds_dict in [
    ('Val',  y_val_np,  {'RF': rf_val,  'XGB': xgb_val,  'MLP (base)': mlp_val,  'Weighted Ensemble': we_val,  'Hybrid (RF+XGB+ResidualMLP)': hybrid_val}),
    ('Test', y_test_np, {'RF': rf_test, 'XGB': xgb_test, 'MLP (base)': mlp_test, 'Weighted Ensemble': we_test, 'Hybrid (RF+XGB+ResidualMLP)': hybrid_test}),
]:
    for model_name, preds in preds_dict.items():
        rows.append(eval_metrics(y_true, preds, model_name, split))

metrics_df = pd.DataFrame(rows)
metrics_df[['MAE', 'RMSE', 'R²']] = metrics_df[['MAE', 'RMSE', 'R²']].round(4)
print('\n' + '='*70)
print('COMPREHENSIVE MODEL COMPARISON — All 5 Models')
print('='*70)
print(metrics_df.to_string(index=False))
print('='*70)

In [ ]:
# ── Scatter plots: Predicted vs Actual (Test set, all 5 models) ───────────
test_preds_all = {
    'Random Forest': rf_test,
    'XGBoost': xgb_test,
    'MLP (base)': mlp_test,
    'Weighted Ensemble': we_test,
    'Hybrid (ResidualMLP)': hybrid_test,
}

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()
colors = ['#2196F3', '#FF9800', '#9C27B0', '#4CAF50', '#F44336']

for ax, (model_name, preds), color in zip(axes, test_preds_all.items(), colors):
    m = compute_regression_metrics(y_test_np, preds)
    ax.scatter(y_test_np, preds, alpha=0.3, s=5, color=color, rasterized=True)
    min_v, max_v = y_test_np.min(), y_test_np.max()
    ax.plot([min_v, max_v], [min_v, max_v], 'k--', lw=1.5, label='Perfect')
    ax.set_title(f'{model_name}\nMAE={m["mae"]:.3f}  RMSE={m["rmse"]:.3f}  R²={m["r2"]:.4f}', fontsize=11)
    ax.set_xlabel('Actual CCME WQI'); ax.set_ylabel('Predicted CCME WQI')
    ax.legend(fontsize=8)

axes[-1].set_visible(False)
plt.suptitle('Phase 4 — Predicted vs Actual (Test Set)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'hybrid_scatter_plots.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Scatter plots saved')

In [ ]:
# ── Residual plots (prediction error distribution) ─────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for ax, (model_name, preds), color in zip(axes, test_preds_all.items(), colors):
    errors = y_test_np - preds
    ax.hist(errors, bins=60, color=color, alpha=0.7, edgecolor='white')
    ax.axvline(0, color='black', lw=1.5, ls='--')
    ax.axvline(np.mean(errors), color='red', lw=1.5, ls='-', label=f'Mean={np.mean(errors):.3f}')
    ax.set_title(f'{model_name}\nStd={np.std(errors):.3f}', fontsize=11)
    ax.set_xlabel('Prediction Error (y − ŷ)'); ax.set_ylabel('Count')
    ax.legend(fontsize=8)

axes[-1].set_visible(False)
plt.suptitle('Phase 4 — Residual Error Distributions (Test Set)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'hybrid_residual_plots.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Residual error plots saved')

## Section 6 — Model Export & Summary

In [ ]:
# ── Save hybrid_config.json ────────────────────────────────────────────────
config_path = OUTPUT_DIR / 'hybrid_config.json'
with open(config_path, 'w') as f:
    json.dump(hybrid_cfg.to_dict(), f, indent=2)
print(f'✓ hybrid_config.json saved → {config_path}')

In [ ]:
# ── Save weighted_ensemble_weights.json ────────────────────────────────────
we_weights_path = OUTPUT_DIR / 'weighted_ensemble_weights.json'
with open(we_weights_path, 'w') as f:
    json.dump(weighted_ens.to_dict(), f, indent=2)
print(f'✓ weighted_ensemble_weights.json saved → {we_weights_path}')

In [ ]:
# ── Save full hybrid checkpoint ────────────────────────────────────────────
test_metrics_hybrid = compute_regression_metrics(y_test_np, hybrid_test)
save_hybrid_checkpoint(
    model    = hybrid_model,
    config   = hybrid_cfg,
    path     = MODELS_DIR / 'hybrid_ensemble_checkpoint.pt',
    metadata = {
        'best_val_mae'  : float(res_history['best_val_mae']),
        'test_mae'      : test_metrics_hybrid['mae'],
        'test_rmse'     : test_metrics_hybrid['rmse'],
        'test_r2'       : test_metrics_hybrid['r2'],
    },
)

In [ ]:
# ── Save test predictions parquet ──────────────────────────────────────────
test_preds_df = pd.DataFrame({
    'y_true'              : y_test_np,
    'rf_pred'             : rf_test,
    'xgb_pred'            : xgb_test,
    'mlp_pred'            : mlp_test,
    'weighted_ensemble'   : we_test,
    'hybrid_residual_mlp' : hybrid_test,
})
preds_path = OUTPUT_DIR / 'hybrid_test_predictions.parquet'
test_preds_df.to_parquet(preds_path, index=False)
print(f'✓ Test predictions saved → {preds_path}  ({len(test_preds_df):,} rows)')

In [ ]:
# ── Final comparison table ─────────────────────────────────────────────────
test_rows = metrics_df[metrics_df['Split'] == 'Test'].copy()
print('\n' + '='*65)
print('FINAL TEST SET COMPARISON — Phase 4 Hybrid Models')
print('='*65)
print(test_rows[['Model', 'MAE', 'RMSE', 'R²']].to_string(index=False))
print('='*65)

best_model_row = test_rows.loc[test_rows['MAE'].idxmin()]
print(f'\n🏆 Best model by MAE : {best_model_row["Model"]} (MAE={best_model_row["MAE"]:.4f})')

## Summary

### Phase 4 Key Findings

**Approach**: Two ensemble strategies were evaluated:
1. **Weighted Ensemble (RF=0.4, XGB=0.4, MLP=0.2)**: Simple linear combination leveraging all three base models.
2. **Hybrid ResidualMLP**: Tree ensemble base predictions corrected by a lightweight 3-layer MLP (54→256→128→64→1) trained on OOF residuals.

**OOF Training**: 5-fold cross-validation on the training set ensures the residual MLP never sees the same samples for both base prediction and residual correction — eliminating target leakage.

**Architecture**: `ResidualMLPNet` takes `[53-dim features ∥ 1-dim base_pred]` = 54-dim input and predicts the signed residual correction. Final prediction = base_pred + residual_correction.

**Artifacts saved**:
- `data/GeneratedOutput/04.HybridModel/oof_predictions.parquet`
- `data/GeneratedOutput/04.HybridModel/hybrid_config.json`
- `data/GeneratedOutput/04.HybridModel/weighted_ensemble_weights.json`
- `data/GeneratedOutput/04.HybridModel/hybrid_test_predictions.parquet`
- `Phase_2_Deep_Learning/models/residual_mlp_checkpoint.pt`
- `Phase_2_Deep_Learning/models/hybrid_ensemble_checkpoint.pt`